# 7.1 候选模型比较

在 3.1 固定的验证集上，红酒、白酒分别比较两个预定候选（Ridge、随机森林）和基线（训练集 quality 中位数）：
MAE 改善多少、RMSE 有没有恶化、重抽样区间是否整体在 0 以下。最终评估集一行都不碰。


In [1]:
import hashlib
import json
import platform
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge

import dsflow

ROOT = Path.cwd()
while not (ROOT / "dsflow.yaml").is_file():
    ROOT = ROOT.parent
STEP = ROOT / "steps/07_模型选择与训练/7.1_候选模型比较"
OUT = STEP / "outputs"
(OUT / "models").mkdir(parents=True, exist_ok=True)
S31 = ROOT / "steps/03_数据划分/3.1_预测任务定义_固定划分与验证基线/outputs"
FEATURES = ["fixed acidity", "volatile acidity", "citric acid", "residual sugar", "chlorides",
            "free sulfur dioxide", "total sulfur dioxide", "density", "pH", "sulphates", "alcohol"]
WINES = {"red": "红酒", "white": "白酒"}
SEED = 20260914
DRAWS = 2000

run = dsflow.start_run("7.1", project=ROOT,
                       hypothesis="用 11 个理化指标的 Ridge 或随机森林，验证集 MAE 低于训练集中位数基线，且重抽样区间上界小于 0")
raw = {}
for w, name in WINES.items():
    path = ROOT / f"data/winequality-{w}.csv"
    run.log_input(path, name=name)
    raw[w] = pd.read_csv(path, sep=";").assign(source_row=lambda d: np.arange(1, len(d) + 1))
run.log_input(S31 / "split_assignments.csv", name="划分表")
splits = pd.read_csv(S31 / "split_assignments.csv")
baseline_ref = json.loads((S31 / "baseline_validation.json").read_text(encoding="utf-8"))
INPUT_FILES = ["data/winequality-red.csv", "data/winequality-white.csv", "data/winequality.names",
               "steps/01_数据预处理/1.1_原始数据理解与质量核验/outputs/data_profile.json",
               "steps/02_EDA/2.1_理化指标与quality的探索分析/outputs/eda.json",
               "steps/03_数据划分/3.1_预测任务定义_固定划分与验证基线/outputs/split_assignments.csv",
               "steps/03_数据划分/3.1_预测任务定义_固定划分与验证基线/outputs/split_summary.json",
               "steps/03_数据划分/3.1_预测任务定义_固定划分与验证基线/outputs/baseline_validation.json"]
fingerprint = lambda rel: {"bytes": (ROOT / rel).stat().st_size, "sha256": hashlib.sha256((ROOT / rel).read_bytes()).hexdigest()}
inputs_before = {rel: fingerprint(rel) for rel in INPUT_FILES}
print(f"红酒 {len(raw['red']):,} 行，白酒 {len(raw['white']):,} 行；划分表 {len(splits):,} 行")
print(f"库版本：python {platform.python_version()}，numpy {np.__version__}，pandas {pd.__version__}，scikit-learn {sklearn.__version__}")


红酒 1,599 行，白酒 4,898 行；划分表 6,497 行
库版本：python 3.14.5，numpy 2.5.3，pandas 3.0.5，scikit-learn 1.9.1


In [2]:
data = {}
for w, name in WINES.items():
    df = raw[w].merge(splits[splits["wine"] == w].drop(columns="wine"), on="source_row", how="left", validate="one_to_one")
    assert df["split"].notna().all(), f"{name}有原始行没在划分表里"
    assert (df.groupby("feature_group_id")["split"].nunique() == 1).all(), f"{name}有特征组合跨划分"
    train = df[df["split"] == "training"].sort_values("source_row").reset_index(drop=True)
    valid = df[df["split"] == "validation"].sort_values("source_row").reset_index(drop=True)
    sealed = int((df["split"] == "final_evaluation").sum())
    ref = baseline_ref["files"][w]
    assert (len(train), len(valid)) == (ref["training_rows"], ref["validation_rows"]), f"{name}训练 / 验证行数和 3.1 不一致"
    median = float(np.median(train["quality"]))
    base_err = np.abs(valid["quality"].to_numpy(float) - median)
    base_mae, base_rmse = float(base_err.mean()), float(np.sqrt((base_err ** 2).mean()))
    assert abs(base_mae - ref["validation_mae"]) < 1e-10 and abs(base_rmse - ref["validation_rmse"]) < 1e-10, f"{name}基线和 3.1 对不上"
    data[w] = {"train": train, "valid": valid, "median": median, "base_mae": base_mae, "base_rmse": base_rmse, "sealed": sealed}
    print(f"{name}：训练 {len(train):,} 行、验证 {len(valid):,} 行（{valid['feature_group_id'].nunique()} 个特征组合）、最终评估 {sealed:,} 行封存；"
          f"基线预测 {median}，验证 MAE {base_mae:.4f}、RMSE {base_rmse:.4f}，和 3.1 一致")


红酒：训练 950 行、验证 331 行（274 个特征组合）、最终评估 318 行封存；基线预测 6.0，验证 MAE 0.6435、RMSE 0.8777，和 3.1 一致
白酒：训练 2,921 行、验证 983 行（794 个特征组合）、最终评估 994 行封存；基线预测 6.0，验证 MAE 0.6368、RMSE 0.9033，和 3.1 一致


In [3]:
RF_PARAMS = {"n_estimators": 300, "min_samples_leaf": 5, "max_features": 1.0, "random_state": SEED, "n_jobs": 1}
RIDGE_PARAMS = {"alpha": 1.0, "fit_intercept": True}
run.log_params({"候选": ["Ridge", "随机森林"], "Ridge": RIDGE_PARAMS, "随机森林": RF_PARAMS,
                "标准化": "只用训练行的均值与标准差（ddof=0），只给 Ridge", "预测": "原始连续输出，不取整、不截断",
                "scikit-learn": sklearn.__version__, "numpy": np.__version__, "重抽样": {"次数": DRAWS, "种子": SEED, "生成器": "PCG64"}})
pred_rows, summary = [], {}
for w, name in WINES.items():
    d = data[w]
    Xtr, ytr = d["train"][FEATURES].to_numpy(float), d["train"]["quality"].to_numpy(float)
    Xva, yva = d["valid"][FEATURES].to_numpy(float), d["valid"]["quality"].to_numpy(float)
    mean, std = Xtr.mean(axis=0), Xtr.std(axis=0)
    (OUT / f"scaler_{w}.json").write_text(json.dumps({"features": FEATURES, "mean": mean.tolist(), "std": std.tolist(), "ddof": 0,
                                                      "fitted_on": "training rows only"}, ensure_ascii=False, indent=1), encoding="utf-8")
    ridge = Ridge(**RIDGE_PARAMS).fit((Xtr - mean) / std, ytr)
    forest = RandomForestRegressor(**RF_PARAMS).fit(Xtr, ytr)
    joblib.dump({"scaler": {"mean": mean, "std": std}, "model": ridge, "features": FEATURES}, OUT / "models" / f"{w}_Ridge.joblib")
    joblib.dump({"model": forest, "features": FEATURES}, OUT / "models" / f"{w}_随机森林.joblib")
    preds = {"Ridge": ridge.predict((Xva - mean) / std), "随机森林": forest.predict(Xva)}
    summary[w] = {"name": name, "baseline": {"prediction": d["median"], "mae": d["base_mae"], "rmse": d["base_rmse"]},
                  "validation_rows": len(d["valid"]), "validation_feature_groups": int(d["valid"]["feature_group_id"].nunique()),
                  "training_rows": len(d["train"]), "final_evaluation_rows_sealed": d["sealed"], "candidates": {}}
    for cand, p in preds.items():
        assert np.isfinite(p).all()
        err = np.abs(yva - p)
        mae, rmse = float(err.mean()), float(np.sqrt((err ** 2).mean()))
        summary[w]["candidates"][cand] = {"mae": mae, "rmse": rmse, "mae_minus_baseline": mae - d["base_mae"], "rmse_minus_baseline": rmse - d["base_rmse"]}
        run.log_metrics({f"{name}_{cand}_MAE_验证": mae, f"{name}_{cand}_RMSE_验证": rmse})
        for i in range(len(yva)):
            pred_rows.append({"wine": w, "source_row": int(d["valid"]["source_row"][i]), "feature_group_id": d["valid"]["feature_group_id"][i],
                              "candidate": cand, "quality": float(yva[i]), "prediction": float(p[i]), "baseline_prediction": d["median"],
                              "abs_error": float(err[i]), "baseline_abs_error": float(abs(yva[i] - d["median"]))})
    run.log_metrics({f"{name}_基线_MAE_验证": d["base_mae"], f"{name}_基线_RMSE_验证": d["base_rmse"]})
predictions = pd.DataFrame(pred_rows)
predictions.to_csv(OUT / "validation_predictions.csv", index=False)
table = pd.DataFrame([{"酒类": s["name"], "候选": c, "验证 MAE": round(v["mae"], 4), "基线 MAE": round(s["baseline"]["mae"], 4),
                       "MAE 差": round(v["mae_minus_baseline"], 4), "验证 RMSE": round(v["rmse"], 4), "基线 RMSE": round(s["baseline"]["rmse"], 4)}
                      for s in summary.values() for c, v in s["candidates"].items()])
print(table.to_string(index=False))


酒类    候选  验证 MAE  基线 MAE   MAE 差  验证 RMSE  基线 RMSE
红酒 Ridge  0.5194  0.6435 -0.1241   0.6713   0.8777
红酒  随机森林  0.5051  0.6435 -0.1384   0.6415   0.8777
白酒 Ridge  0.5814  0.6368 -0.0555   0.7586   0.9033
白酒  随机森林  0.5474  0.6368 -0.0894   0.7026   0.9033


In [4]:
draws, diff_rows = {}, []
for w, name in WINES.items():
    valid = data[w]["valid"]
    ids = sorted(valid["feature_group_id"].unique())
    G = len(ids)
    pos = {g: i for i, g in enumerate(ids)}
    rows_per_group = np.bincount(valid["feature_group_id"].map(pos), minlength=G)
    err_sum = {"基线": np.bincount(valid["feature_group_id"].map(pos), weights=np.abs(valid["quality"] - data[w]["median"]), minlength=G)}
    for cand in ("Ridge", "随机森林"):
        p = predictions[(predictions["wine"] == w) & (predictions["candidate"] == cand)].set_index("source_row").loc[valid["source_row"]]
        err_sum[cand] = np.bincount(valid["feature_group_id"].map(pos), weights=p["abs_error"].to_numpy(), minlength=G)
    rng = np.random.Generator(np.random.PCG64(SEED))
    idx = rng.integers(0, G, size=(DRAWS, G))
    draws[w] = {"ordered_feature_group_ids": ids, "indices": idx.tolist()}
    for cand in ("Ridge", "随机森林"):
        diffs = np.empty(DRAWS)
        for b in range(DRAWS):
            times = np.bincount(idx[b], minlength=G)
            n = float((times * rows_per_group).sum())
            diffs[b] = (times * err_sum[cand]).sum() / n - (times * err_sum["基线"]).sum() / n
        lo, hi = (float(x) for x in np.percentile(diffs, [2.5, 97.5], method="linear"))
        summary[w]["candidates"][cand]["mae_difference_interval"] = {"lower_2_5": lo, "upper_97_5": hi, "draws": DRAWS}
        run.log_metrics({f"{name}_{cand}_MAE差_区间上界": hi, f"{name}_{cand}_MAE差_区间下界": lo})
        diff_rows += [{"draw": b, "wine": w, "candidate": cand, "mae_minus_baseline": float(diffs[b])} for b in range(DRAWS)]
        print(f"{name} {cand}：候选 MAE − 基线 MAE 的 2.5%～97.5% 区间 [{lo:.4f}, {hi:.4f}]（{G} 个验证特征组合，重抽 {DRAWS} 次）")
(OUT / "bootstrap_draws.json").write_text(json.dumps({"seed": SEED, "generator": "numpy.random.Generator(numpy.random.PCG64(seed))",
    "draws": DRAWS, "rule": "每次从 G 个验证特征组合等概率有放回抽 G 个索引；抽中几次其全部验证行计几次；按抽中后的原始行数求 MAE；两个候选与基线共用同一批索引",
    "wines": draws}, ensure_ascii=False), encoding="utf-8")
pd.DataFrame(diff_rows).to_csv(OUT / "bootstrap_mae_differences.csv", index=False)


红酒 Ridge：候选 MAE − 基线 MAE 的 2.5%～97.5% 区间 [-0.1973, -0.0542]（274 个验证特征组合，重抽 2000 次）
红酒 随机森林：候选 MAE − 基线 MAE 的 2.5%～97.5% 区间 [-0.2130, -0.0663]（274 个验证特征组合，重抽 2000 次）
白酒 Ridge：候选 MAE − 基线 MAE 的 2.5%～97.5% 区间 [-0.0937, -0.0191]（794 个验证特征组合，重抽 2000 次）
白酒 随机森林：候选 MAE − 基线 MAE 的 2.5%～97.5% 区间 [-0.1335, -0.0458]（794 个验证特征组合，重抽 2000 次）


In [5]:
registered = []
for w, name in WINES.items():
    valid = data[w]["valid"]
    by_q = {}
    for q, g in valid.groupby("quality"):
        entry = {"rows": int(len(g)), "feature_groups": int(g["feature_group_id"].nunique()),
                 "baseline_mae": float(np.abs(g["quality"] - data[w]["median"]).mean())}
        for cand in ("Ridge", "随机森林"):
            p = predictions[(predictions["wine"] == w) & (predictions["candidate"] == cand) & predictions["source_row"].isin(g["source_row"])]
            entry[f"{cand}_mae"] = float(p["abs_error"].mean())
        by_q[str(int(q))] = entry
    summary[w]["by_quality"] = by_q
    for cand, v in summary[w]["candidates"].items():
        s = summary[w]
        v["meets_threshold"] = bool(v["mae"] < s["baseline"]["mae"] and v["mae_difference_interval"]["upper_97_5"] < 0 and v["rmse"] <= s["baseline"]["rmse"])
        if v["meets_threshold"]:
            registered.append(f"{name} {cand}")
inputs_after = {rel: fingerprint(rel) for rel in INPUT_FILES}
assert inputs_after == inputs_before, "输入文件在执行过程中变了"
comparison = {"threshold": "验证 MAE 低于基线，且候选 MAE − 基线 MAE 的 97.5% 分位数小于 0，且 RMSE 不高于基线；只用于登记进一步验证的候选，不是生产收益门槛",
              "features": FEATURES, "ridge_params": RIDGE_PARAMS, "forest_params": RF_PARAMS,
              "versions": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "scikit-learn": sklearn.__version__},
              "inputs_before": inputs_before, "inputs_after": inputs_after, "inputs_unchanged": True,
              "registered_candidates": registered, "files": summary}
(OUT / "model_comparison.json").write_text(json.dumps(comparison, ensure_ascii=False, indent=1), encoding="utf-8")
run.log_artifact(OUT / "model_comparison.json", purpose="每类酒的基线、两个候选的 MAE / RMSE、相对基线的差、重抽样区间、按 quality 分值的误差、登记判定", kind="table")
run.log_artifact(OUT / "validation_predictions.csv", purpose="每个候选对每条验证行的连续预测、真值、基线预测与双方误差", kind="table")
run.log_artifact(OUT / "bootstrap_draws.json", purpose="重抽样的有序组合 ID 与全部抽样索引，供独立重算区间", kind="other")
run.log_artifact(OUT / "bootstrap_mae_differences.csv", purpose="2,000 次重抽样里每次每候选的 MAE 配对差", kind="table")
for w in WINES:
    for cand in ("Ridge", "随机森林"):
        run.log_artifact(OUT / "models" / f"{w}_{cand}.joblib", purpose=f"{WINES[w]}的 {cand} 候选模型文件，只用训练行拟合，供 9.1 只评一次", kind="model")
lines = []
for w, name in WINES.items():
    s = summary[w]
    for cand, v in s["candidates"].items():
        ci = v["mae_difference_interval"]
        lines.append(f"{name} {cand}：验证 MAE {v['mae']:.4f}（基线 {s['baseline']['mae']:.4f}，差 {v['mae_minus_baseline']:+.4f}，区间 [{ci['lower_2_5']:.4f}, {ci['upper_97_5']:.4f}]），"
                     f"RMSE {v['rmse']:.4f}（基线 {s['baseline']['rmse']:.4f}），{'满足' if v['meets_threshold'] else '不满足'}登记门槛")
print("\n".join(lines))
print(pd.DataFrame([{"酒类": WINES[w], "quality": q, **e} for w in WINES for q, e in summary[w]["by_quality"].items()]).round(4).to_string(index=False))


红酒 Ridge：验证 MAE 0.5194（基线 0.6435，差 -0.1241，区间 [-0.1973, -0.0542]），RMSE 0.6713（基线 0.8777），满足登记门槛
红酒 随机森林：验证 MAE 0.5051（基线 0.6435，差 -0.1384，区间 [-0.2130, -0.0663]），RMSE 0.6415（基线 0.8777），满足登记门槛
白酒 Ridge：验证 MAE 0.5814（基线 0.6368，差 -0.0555，区间 [-0.0937, -0.0191]），RMSE 0.7586（基线 0.9033），满足登记门槛
白酒 随机森林：验证 MAE 0.5474（基线 0.6368，差 -0.0894，区间 [-0.1335, -0.0458]），RMSE 0.7026（基线 0.9033），满足登记门槛
酒类 quality  rows  feature_groups  baseline_mae  Ridge_mae  随机森林_mae
红酒       3     2               2           3.0     2.1074    2.0563
红酒       4    11              11           2.0     1.3419    1.2016
红酒       5   135             116           1.0     0.4036    0.3587
红酒       6   137             107           0.0     0.4124    0.4495
红酒       7    42              34           1.0     0.8315    0.7735
红酒       8     4               4           2.0     1.7593    1.8450
白酒       3     4               4           3.0     2.2681    2.5496
白酒       4    33              31           2.0     1.4718    1.3797
白酒    

In [6]:
conclusion = ("满足登记门槛的候选：" + ("、".join(registered) if registered else "没有") + "。" +
              "；".join(f"{WINES[w]} {c} 验证 MAE {v['mae']:.4f} 对基线 {summary[w]['baseline']['mae']:.4f}" for w in WINES for c, v in summary[w]["candidates"].items()))
run.set_conclusion(conclusion, validity="有效")
run.end()
print(conclusion)


满足登记门槛的候选：红酒 Ridge、红酒 随机森林、白酒 Ridge、白酒 随机森林。红酒 Ridge 验证 MAE 0.5194 对基线 0.6435；红酒 随机森林 验证 MAE 0.5051 对基线 0.6435；白酒 Ridge 验证 MAE 0.5814 对基线 0.6368；白酒 随机森林 验证 MAE 0.5474 对基线 0.6368
